# Modelling a repurchase agreement (repo) and handling instrument events

This notebook demonstrates the concepts described in the following KBs:

* Mastering a `FlexibleRepo` instrument, establishing a position, and performing a valuation: https://support.lusid.com/docs/modelling-repurchase-agreements-in-lusid
* Handling instrument events: https://support.lusid.com/docs/handling-instrument-events-for-repurchase-agreements

Two `FlexibleRepo` instruments are created, with separate positions in two portfolios. The two instruments are identical except one is a `Buyer` and the other a `Seller`:

* Both are `TermRepo` by default but the code for `OpenRepo` is present and commented out.
* Underlying is a simple fixed-rate `Bond` with monthly coupon payments.
* Transaction date: 01 Jan 2025 (Note the 'purchase' transaction of a `FlexibleRepo` **must be before** the instrument start date)
* Instrument start date: 02 Jan 2025
* Maturity date: 31 Mar 2025
* Random valuation date: 15 Feb 2025
* Accrual basis: Act 365
* Collateral value: £1,100,000
* Purchase price: £1,000,000
* Margin: 1.1
* Haircut: 0.091
* Repurchase price: £1,012,054.79
* Repo rate : 5%

## Setup

In [1]:
import os
import pandas as pd
import json
import uuid
from IPython.core.display import HTML
import logging
from datetime import datetime, timezone, timedelta
logging.basicConfig(level = logging.INFO)

import finbourne.sdk.services.lusid.api as la
import finbourne.sdk.services.lusid.models as lm

from finbourne.sdk.extensions import SyncApiClientFactory, RefreshingToken
from finbourne.sdk.exceptions import ApiException
from finbourne_sdk_utils.pandas_utils.lusid_pandas import lusid_response_to_data_frame
from finbourne_sdk_utils.lpt.lpt import to_date

# Set pandas display options
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.options.display.float_format = "{:,.2f}".format

# Authenticate to SDK
# Run the Notebook in Jupyterhub for your LUSID domain and authenticate automatically
secrets_path = os.getenv("FBN_SECRETS_PATH")
# Run the Notebook locally using a secrets file (see https://support.lusid.com/docs/how-do-i-use-an-api-access-token-with-the-lusid-sdk)
if secrets_path is None:
    secrets_path = os.path.join(os.path.dirname(os.getcwd()), "secrets.json")

# Initiate an API Factory which is the client side object for interacting with LUSID APIs
api_factory = SyncApiClientFactory(
    access_token=RefreshingToken(),
    secrets_path=secrets_path,
    app_name="LusidJupyterNotebook"
)
    
# Confirm success by printing SDK version
api_status = pd.DataFrame(api_factory.build(la.ApplicationMetadataApi).get_lusid_versions().to_dict())
display(api_status)

,apiVersion,buildVersion,excelVersion,links
0,v0,0.6.15978.0,0.5.3666,"{'relation': 'RequestLogs', 'href': 'https://j..."


In [2]:
# Build all the required APIs
try:
    instruments_api = api_factory.build(la.InstrumentsApi)
    instrument_events_api = api_factory.build(la.InstrumentEventsApi)
    instrument_event_type_api = api_factory.build(la.InstrumentEventTypesApi)
    aggregation_api = api_factory.build(la.AggregationApi)
    recipe_api = api_factory.build(la.ConfigurationRecipeApi)
    quotes_api = api_factory.build(la.QuotesApi)
    property_definition_api = api_factory.build(la.PropertyDefinitionsApi)
    transaction_portfolios_api = api_factory.build(la.TransactionPortfoliosApi)
    portfolios_api = api_factory.build(la.PortfoliosApi)
    transaction_config_api = api_factory.build(la.TransactionConfigurationApi)
    print("All APIs built correctly")
except ApiException as e:
    print(e)

All APIs built correctly


## Create a scope and code for entities in the Notebook

Keep data segregated from other data in LUSID.

In [3]:
# Create a scope and code to segregate data in this Notebook from others
module_scope = "FBNTutorials"
module_code = "FlexRepo"
print(f"'{module_scope}\\{module_code}' scope and code created.")

'FBNTutorials\FlexRepo' scope and code created.


## Create property types

Create a SHK to split out underlying bond coupon cash in holding reports from other cash balances.

In [4]:
def create_property_type(property_domain, property_scope, property_code, data_type):
    property_type_request = lm.CreatePropertyDefinitionRequest(
        domain = property_domain,
        scope = property_scope,
        code = property_code,
        display_name = property_code,
        data_type_id = lm.ResourceId(scope = "system", code = data_type)
    )

    try:
        property_type_response = property_definition_api.create_property_definition(
            create_property_definition_request = property_type_request
        )
        print(f"Property type created with the following key: {property_type_response.key}")
        return property_type_response.key
    except ApiException as e:
        if json.loads(e.body)["name"] == "PropertyAlreadyExists":
            logging.info(
                f"Property type with the following key already exists: {property_type_request.domain}/{property_type_request.scope}/{property_type_request.code}"
            )  
        return f"{property_type_request.domain}/{property_type_request.scope}/{property_type_request.code}"

In [5]:
bond_coupon_shk = create_property_type("Transaction", "SHKs", "BondCollateralCoupons", "string")

INFO:root:Property type with the following key already exists: Transaction/SHKs/BondCollateralCoupons


## Create transaction types and sides

Required for both explicit purchase transactions and for transactions automatically generated by instrument events.

All created in a custom transaction type scope, which must be registered with the portfolios in which transactions are loaded.

In [6]:
def check_TT(tt, scope):
    try:
        tt_response = transaction_config_api.get_transaction_type(source = f"default", type = tt, scope=scope)
        print(f"\n{tt} transaction type:")
        display(lusid_response_to_data_frame(tt_response.aliases))
        display(lusid_response_to_data_frame(tt_response.movements))
        display(lusid_response_to_data_frame(tt_response.calculations))
    except ApiException as e:
        print(e)
        
def check_side(side, scope):
    try:
        side_response = transaction_config_api.get_side_definition(scope = scope, side = side)
        print(f"\n{side} side:")
        side_response_df = lusid_response_to_data_frame(side_response).transpose()
        side_response_df.drop(side_response_df.filter(regex='links').columns, axis=1, inplace=True)
        display(side_response_df)  
    except ApiException as e:
        print(e)

### Create sides

Must be created before transaction types. Includes recreating the built-in `Side1` and `Side2` in the custom transaction type scope.

In [7]:
# Recreate Side1 in custom scope
side_definition = lm.SideDefinitionRequest(
    security = "Txn:LusidInstrumentId",
    currency = "Txn:TradeCurrency",
    rate = "Txn:TradeToPortfolioRate",
    units = "Txn:Units",
    amount = "Txn:TradeAmount"
)

try:
    response = transaction_config_api.set_side_definition(
        side = "Side1",
        scope= f"{module_scope}{module_code}",
        side_definition_request = side_definition
    )
    print("Success")
except ApiException as e:
    print(e)

Success


In [8]:
# Recreate Side2 in custom scope
side_definition = lm.SideDefinitionRequest(
    security = "Txn:SettleCcy",
    currency = "Txn:SettlementCurrency",
    rate = "SettledToPortfolioRate",
    units = "Txn:TotalConsideration",
    amount = "Txn:TotalConsideration"
)

try:
    response = transaction_config_api.set_side_definition(
        side = "Side2",
        scope= f"{module_scope}{module_code}",
        side_definition_request = side_definition
    )
    print("Success")
except ApiException as e:
    print(e)

Success


In [9]:
# Define custom 'Side1EnterRepo' to handle units for purchase transaction type
side_definition = lm.SideDefinitionRequest(
    security = "Txn:LusidInstrumentId",
    currency = "Txn:TradeCurrency",
    rate = "Txn:TradeToPortfolioRate",
    units = "Txn:Units",
    amount = "0", 
)

try:
    response = transaction_config_api.set_side_definition(
        side = "Side1EnterRepo",
        scope= f"{module_scope}{module_code}",
        side_definition_request = side_definition
    )
    print("Success")
except ApiException as e:
    print(e)

Success


In [10]:
# Define custom 'Side1EnterRepoCost' to handle cost for purchase transaction type
side_definition = lm.SideDefinitionRequest(
    security = "Txn:LusidInstrumentId",
    currency = "Txn:TradeCurrency",
    rate = "Txn:TradeToPortfolioRate",
    units = "0",
    amount = "Txn:TotalConsideration", 
)

try:
    response = transaction_config_api.set_side_definition(
        side = "Side1EnterRepoCost",
        scope= f"{module_scope}{module_code}",
        side_definition_request = side_definition
    )
    print("Success")
except ApiException as e:
    print(e)

Success


In [11]:
# Define custom 'Side1RepoCashTransfer' side to handle repurchase price for FlexibleRepoCashFlowEvent
side_definition = lm.SideDefinitionRequest(
    security = "Txn:LusidInstrumentId",
    currency = "Txn:TradeCurrency",
    rate = "Txn:TradeToPortfolioRate",
    units = "0",
    amount = "Txn:TotalConsideration", 
)

try:
    response = transaction_config_api.set_side_definition(
        side = "Side1RepoCashTransfer",
        scope= f"{module_scope}{module_code}",
        side_definition_request = side_definition
    )
    print("Success")
except ApiException as e:
    print(e)

Success


In [12]:
# Define custom 'Side1RepoInterestPayment' side to report interest for FlexibleRepoInterestPaymentEvent
side_definition = lm.SideDefinitionRequest(
    security = "Txn:LusidInstrumentId",
    currency = "Txn:TradeCurrency",
    rate = "Txn:TradeToPortfolioRate",
    units = "Txn:Units",
    amount = "Txn:TotalConsideration", 
)

try:
    response = transaction_config_api.set_side_definition(
        side = "Side1RepoInterestPayment",
        scope= f"{module_scope}{module_code}",
        side_definition_request = side_definition
    )
    print("Success")
except ApiException as e:
    print(e)

Success


### Create transaction types

Note transaction type calculations are deliberately *not* used for `FlexibleRepo`.

#### Create `EnterRepo` transaction type (to establish positions)

Three movements, two of which use custom sides.

In [13]:
transaction_type_definition = lm.TransactionTypeRequest(
    aliases = [
        lm.TransactionTypeAlias(
            type = "EnterRepo",
            description = "Open the contract",
            transaction_class = "Basic",
            transaction_roles = "AllRoles",
            is_default = False
        )
    ],
    movements = [
        lm.TransactionTypeMovement(
            movement_types = "StockMovement",
            side = "Side1EnterRepo",
            direction = 1
        ),
        lm.TransactionTypeMovement(
            movement_types = "CashReceivable",
            side = "Side2",
            direction = 1
        ),
        lm.TransactionTypeMovement(
            movement_types = "StockMovement",
            side = "Side1EnterRepoCost",
            direction = -1
        )
    ],
)

try:
    response = transaction_config_api.set_transaction_type(
        source = "default",
        scope = f"{module_scope}{module_code}",
        # Specify the primary alias name
        type = "EnterRepo",
        transaction_type_request = transaction_type_definition
    )
    print("Success")
except ApiException as e:
    print(e)

Success


### Create `FlexibleRepoCashTransfer` transaction type (to handle `FlexibleRepoCashFlowEvent`)

The `StockMovement` uses a custom side.

In [14]:
transaction_type_definition = lm.TransactionTypeRequest(
    aliases = [
        lm.TransactionTypeAlias(
            type = "FlexibleRepoCashTransfer",
            description = "Transaction type for transactions automatically generated by FlexibleRepoCashFlowEvent",
            transaction_class = "Basic",
            transaction_roles = "AllRoles",
            is_default = False
        )
    ],
    movements = [
        lm.TransactionTypeMovement(
            name = "Transfer cash",
            movement_types = "CashReceivable",
            side = "Side2",
            direction = 1
        ),
        lm.TransactionTypeMovement(
            name = "Impact cost of stock",
            movement_types = "StockMovement",
            side = "Side1RepoCashTransfer",
            direction = -1
        )
    ]
)
# Replace transaction type in LUSID    
try:
    response = transaction_config_api.set_transaction_type(
        # The source must be 'default' to use the built-in instrument event transaction template
        source = "default",
        scope = f"{module_scope}{module_code}",
        # Specify the primary alias name
        type = "FlexibleRepoCashTransfer",
        transaction_type_request = transaction_type_definition
    )
    print("Success")
except lu.ApiException as e:
    print(e)

Success


### Create `FlexibleRepoCollateralTransfer` transaction type (to handle `FlexibleRepoCollateralEvent`)

In [15]:
transaction_type_definition = lm.TransactionTypeRequest(
    aliases = [
        lm.TransactionTypeAlias(
            type = "FlexibleRepoCollateralTransfer",
            description = "Transaction type for transactions automatically generated by FlexibleRepoCollateralEvent",
            transaction_class = "Basic",
            transaction_roles = "AllRoles",
            is_default = False
        )
    ],
    movements = [
        lm.TransactionTypeMovement(
            name = "Transfer collateral",
            movement_types = "StockMovement",
            side = "Side1",
            direction = 1
        )
    ]
)

try:
    response = transaction_config_api.set_transaction_type(
        # The source must be 'default' to use the built-in instrument event transaction template        
        source = "default",
        scope = f"{module_scope}{module_code}",
        # Specify the primary alias name
        type = "FlexibleRepoCollateralTransfer",
        transaction_type_request = transaction_type_definition
    )
    print("Success")
except lu.ApiException as e:
    print(e)

Success


### Create `FlexibleRepoInterestPayment` transaction type (to handle `FlexibleRepoInterestPaymentEvent`)

The `Carry` movement has a custom side.

In [16]:
transaction_type_definition = lm.TransactionTypeRequest(
    aliases = [
        lm.TransactionTypeAlias(
            type = "FlexibleRepoInterestPayment",
            description = "Transaction type for transactions automatically generated by FlexibleRepoInterestPaymentEvent",
            transaction_class = "Basic",
            transaction_roles = "AllRoles",
            is_default = False
        )
    ],
    movements = [
        lm.TransactionTypeMovement(
            name = "Add repo interest",
            movement_types = "CashAccrual",
            side = "Side2",
            direction = 1
        ),
        lm.TransactionTypeMovement(
            name = "Report repo interest",
            movement_types = "Carry",
            side = "Side1RepoInterestPayment",
            direction = 1
        )
    ]
)
    
try:
    response = transaction_config_api.set_transaction_type(
        # The source must be 'default' to use the built-in instrument event transaction template
        source = "default",
        scope = f"{module_scope}{module_code}",
        # Specify the primary alias name
        type = "FlexibleRepoInterestPayment",
        transaction_type_request = transaction_type_definition
    )
    print("Success")
except lu.ApiException as e:
    print(e)

Success


### Create `Maturity` transaction type (to handle `MaturityEvent`)

In [17]:
transaction_type_definition = lm.TransactionTypeRequest(
    aliases = [
        lm.TransactionTypeAlias(
            type = "Maturity",
            description = "Type for maturity event for various instruments",
            transaction_class = "Basic",
            transaction_roles = "AllRoles",
            is_default = False
        )
    ],
    movements = [
        lm.TransactionTypeMovement(
            name = "Set holding to zero",
            movement_types = "StockMovement",
            direction = -1,
            side = "Side1",
        ),
    ]
)

try:
    response = transaction_config_api.set_transaction_type(
        # The source must be 'default' to use the built-in instrument event transaction template
        source = "default",
        scope = f"{module_scope}{module_code}",
        # Specify the primary alias name
        type = "Maturity",
        transaction_type_request = transaction_type_definition
    )
    print("Success")
except lu.ApiException as e:
    print(e)

Success


### Create `BondCoupon` transaction type (for mandatory `BondCouponEvent`)

This is for the underlying `Bond` collateral instrument while it's part of the `FlexibleRepo`. See https://support.lusid.com/docs/handling-bond-coupon-principal-and-maturity-instrument-lifecycle-events.

In [18]:
transaction_type_definition = lm.TransactionTypeRequest(
    aliases = [
        lm.TransactionTypeAlias(
            type = "BondCoupon",
            description = "Type for bond coupon event",
            transaction_class = "Basic",
            transaction_roles = "AllRoles",
            is_default = False
        )
    ],
    movements = [
        # Create a positive movement that increases a separate cash balance (under SHK) by the gross coupon payment
        lm.TransactionTypeMovement(
            name = "Add coupon to separate cash balance",
            movement_types = "CashAccrual",
            direction = 1,
            side = "Side2",
            mappings = [
                lm.TransactionTypePropertyMapping (
                    property_key = f"{bond_coupon_shk}",
                    set_to = "BondCollateralCoupons",
                )
            ],
        ),
        lm.TransactionTypeMovement(
            name = "Report the coupon as a flow out of the investment",
            movement_types = "Carry",
            direction = 1,
            side = "Side1",
        )
    ]
)

try:
    response = transaction_config_api.set_transaction_type(
        # The source must be 'default' to use the built-in instrument event transaction template
        source = "default",
        scope = f"{module_scope}{module_code}",
        # Specify the primary alias name
        type = "BondCoupon",
        transaction_type_request = transaction_type_definition
    )
    print("Success")
except lu.ApiException as e:
    print(e)

Success


### Create `StockIn` transaction type in custom scope

This is to simply establish a position in the underlying `Bond` instrument in portfolios so collateral units can be transferred.

Note transaction type calculations are enabled to calculate accrual correctly; see https://support.lusid.com/docs/what-is-a-transaction-type-calculation#txnbondinterest-calculation.

In [19]:
transaction_type_definition = lm.TransactionTypeRequest(
    aliases = [
        lm.TransactionTypeAlias(
            type = "StockIn",
            description = "Transfer in",
            transaction_class = "StockTransfers",
            transaction_roles = "Longer",
            is_default = False
        )
    ],
    movements = [
        lm.TransactionTypeMovement(
            movement_types = "StockMovement",
            direction = 1,
            side = "Side1",
        )
    ],
    calculations = [
        lm.TransactionTypeCalculation(
            type = "Txn:BondInterest"
        ),
        lm.TransactionTypeCalculation(
            type = "Txn:GrossConsideration"
        ),
        lm.TransactionTypeCalculation(
            type = "DeriveTotalConsideration",
            formula = "Txn:GrossConsideration"
        )
    ]
)

try:
    response = transaction_config_api.set_transaction_type(
        source = "default",
        scope = f"{module_scope}{module_code}",
        # Specify the primary alias name
        type = "StockIn",
        transaction_type_request = transaction_type_definition
    )
    print("Success")
except lu.ApiException as e:
    print(e)

Success


In [20]:
check_side("Side1", f"{module_scope}{module_code}")
check_side("Side2", f"{module_scope}{module_code}")

check_TT("EnterRepo", f"{module_scope}{module_code}")
check_side("Side1EnterRepo", f"{module_scope}{module_code}")
check_side("Side1EnterRepoCost", f"{module_scope}{module_code}")

check_TT("FlexibleRepoCashTransfer", f"{module_scope}{module_code}")
check_side("Side1RepoCashTransfer", f"{module_scope}{module_code}")

check_TT("FlexibleRepoCollateralTransfer", f"{module_scope}{module_code}")

check_TT("FlexibleRepoInterestPayment", f"{module_scope}{module_code}")
check_side("Side1RepoInterestPayment", f"{module_scope}{module_code}")

check_TT("Maturity", f"{module_scope}{module_code}")

# For underlying bond collateral
check_TT("BondCoupon", f"{module_scope}{module_code}")
check_TT("StockIn", f"{module_scope}{module_code}")


Side1 side:


,side,security,currency,rate,units,amount,notional_amount,current_face
response_values,Side1,Txn:LusidInstrumentId,Txn:TradeCurrency,Txn:TradeToPortfolioRate,Txn:Units,Txn:TradeAmount,0,None



Side2 side:


,side,security,currency,rate,units,amount,notional_amount,current_face
response_values,Side2,Txn:SettleCcy,Txn:SettlementCurrency,SettledToPortfolioRate,Txn:TotalConsideration,Txn:TotalConsideration,0,None



EnterRepo transaction type:


...

""



StockIn transaction type:


,type,description,transaction_class,transaction_roles,is_default
0,StockIn,Transfer in,StockTransfers,Longer,False


,movement_types,side,direction,properties,mappings,movement_options,condition,settlement_mode
0,StockMovement,Side1,1,{},[],[],,Internal


,type,formula
0,Txn:BondInterest,None
1,Txn:GrossConsideration,None
2,DeriveTotalConsideration,Txn:GrossConsideration


## Master instruments

For more information, see https://support.lusid.com/docs/modelling-repurchase-agreements-in-lusid#mastering-an-instrument.

### Set up global numeric variables for `FlexibleRepo`

**`purchasePrice`**:  If `collateralValue` is known, then either specify `haircut` or `margin` and LUSID calculates the purchase price for you. But you can specify an explicit `purchasePrice` if you want.

**`repurchasePrice`**:  If `repoRate` is known, specify it in the `FixedSchedule` or `FloatSchedule` for a `FlexibleRepo` and LUSID calculates the repurchase price for you. But you can specify an explicit `repurchasePrice` if you want.

In [21]:
myCollateralValue = 1100000

myHaircut = 0.09090909090909090909
myMargin = None
myPurchasePrice = None

myRepoRate = 0.05
myRepurchasePrice = None

In [22]:
def master_instrument(assetClass, id, currency, accrualBasis, start, end, repoActor, repoClosure, transferUnits):
    
    if repoClosure == "TermRepo":
        tenor = "1T"
    else:
        tenor = "1W"

    if assetClass == "bond":
        instrument_request = {
            id: lm.InstrumentDefinition(
                name = id,
                identifiers = {"ClientInternal": lm.InstrumentIdValue(value = id)},
                definition = lm.Bond(
                    instrument_type = "Bond", 
                    dom_ccy = currency,
                    start_date = to_date(start),
                    maturityDate=to_date(end),
                    flow_conventions = lm.FlowConventions(
                        currency = currency,
                        payment_frequency = '1M',
                        day_count_convention = accrualBasis,
                        roll_convention = '28',
                        payment_calendars = [],
                        reset_calendars = [],
                    ),
                    principal = 1000,
                    coupon_rate = 0.25,
                )
            )
        }
    if assetClass == "flexrepo":
        instrument_request = {
            id: lm.InstrumentDefinition(
                name = id,
                identifiers = {"ClientInternal": lm.InstrumentIdValue(value = id)},
                definition = lm.FlexibleRepo(
                    instrument_type = "FlexibleRepo", 
                    start_date = to_date(start),
                    maturity_date = to_date(end),   # Required for TermRepo. For OpenRepo, specify an open-ended date
                    buyerOrSeller = repoActor,
                    repoCcy=currency,
                    repoType=repoClosure,
                    collateral = lm.Collateral(
                        buyerReceivesCashflows=False,
                        buyerReceivesCorporateActionPayments=False,
                        collateralInstruments = [
                            lm.CollateralInstrument(
                                units = 1, 
                                instrument = lm.MasteredInstrument(
                                    identifiers={"Instrument/default/ClientInternal": "BondCollateral"},
                                    instrument_type="MasteredInstrument"
                                )
                            )
                        ],
                        collateralValue=myCollateralValue,                
                    ),
                    isCollateralTransferActivated=transferUnits,                   
                    haircut = myHaircut,
                    margin=myMargin,
                    purchasePrice = myPurchasePrice,
                    # Can be single FixedSchedule or FloatSchedule
                    repo_rate_schedules=[lm.FixedSchedule(
                        schedule_type = 'FixedSchedule',
                        start_date = to_date(start),  # Same as flex repo itself
                        maturity_date = to_date(end),  # Same as flex repo itself
                        flow_conventions = lm.FlowConventions(
                            currency = currency,
                            payment_frequency = tenor,  # Must be 1T for TermRepo, same as openRepoRollingPeriod for OpenRepo
                            day_count_convention = accrualBasis,
                            roll_convention = 'None',
                            payment_calendars = [],
                            reset_calendars = [],
                        ),
                        coupon_rate = myRepoRate,
                        notional = 1,
                        payment_currency = currency,
                    )],
                    repurchasePrice = myRepurchasePrice,
                    openRepoRollingPeriod = tenor, # Required for OpenRepo; has no effect on TermRepo
                )
            )
        }
    
    try:
        instrument_response = instruments_api.upsert_instruments(
            request_body = instrument_request,
            scope = f"{module_scope}{module_code}"
        )
        # Return LUID from (only) Instrument object
        return list(instrument_response.values.values())[0].lusid_instrument_id
    except ApiException as e:
        print(e)

In [23]:
luid_dict = {}
luid_dict["BondCollateral"] = master_instrument("bond", "BondCollateral", "GBP", "Act365", "2015-03-28", "2030-03-28", "", "", "")

In [24]:
# Term Repo
luid_dict["Buyer-TermFlexRepo"] = master_instrument("flexrepo", "Buyer-TermFlexRepo", "GBP", "Act365", "2025-01-02", "2025-03-31", "Buyer", "TermRepo", True)
luid_dict["Seller-TermFlexRepo"] = master_instrument("flexrepo", "Seller-TermFlexRepo", "GBP", "Act365", "2025-01-02", "2025-03-31", "Seller", "TermRepo", True)

# Open Repo - note long-dated maturity date
# luid_dict["Buyer-OpenFlexRepo"] = master_instrument("flexrepo", "Buyer-OpenFlexRepo", "GBP", "Act365", "2025-01-02", "2199-12-31", "Buyer", "OpenRepo", True)
# luid_dict["Seller-OpenFlexRepo"] = master_instrument("flexrepo", "Seller-OpenFlexRepo", "GBP", "Act365", "2025-01-02", "2199-12-31", "Seller", "OpenRepo", True)

for k, v in luid_dict.items():
    print(f"{k}: {v}")

BondCollateral: LUID_00003H1O
Buyer-TermFlexRepo: LUID_00003H1P
Seller-TermFlexRepo: LUID_00003H1Q


In [25]:
def list_instrs():
    instr_response = instruments_api.list_instruments(scope=f"{module_scope}{module_code}")
    instr_response_df = lusid_response_to_data_frame(instr_response)
    instr_response_df.drop(instr_response_df.filter(regex='version|href|staged').columns, axis=1, inplace=True)
    display(instr_response_df.transpose())

list_instrs()

,0,1,2
scope,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo
lusid_instrument_id,LUID_00003H1O,LUID_00003H1P,LUID_00003H1Q
name,BondCollateral,Buyer-TermFlexRepo,Seller-TermFlexRepo
identifiers.ClientInternal,BondCollateral,Buyer-TermFlexRepo,Seller-TermFlexRepo
identifiers.LusidInstrumentId,LUID_00003H1O,LUID_00003H1P,LUID_00003H1Q
properties,[],[],[]
instrument_definition.instrument_type,Bond,FlexibleRepo,FlexibleRepo
state,Active,Active,Active
asset_class,Credit,InterestRates,InterestRates
dom_ccy,GBP,GBP,GBP


## Create a recipe

Must be specified as a portfolio recipe to enable instrument events. Can also be used as a valuation recipe.

Note the pricing models have been changed from the defaults for both the `Bond` collateral (to `BondLookupPricer`) and all `FlexibleRepo` instruments (to `FlexibleRepoSimplePricer`).

In [26]:
recipe = lm.ConfigurationRecipe(
    # Put the recipe in the same scope as portfolios
    scope = module_scope,
    # Give the recipe a unique code in the scope
    code = f"{module_code}-FlexibleRepoSimplePricer",
    description = "A recipe to value a repurchase agreement and the underlying collateral",
    market = lm.MarketContext(
        market_rules = [
            # Look up FX spot rates in the LUSID quote store, if needed
            lm.MarketDataKeyRule(
                key = "Fx.CurrencyPair.*",
                supplier = "Lusid",
                data_scope = f"{module_scope}{module_code}",
                quote_type = "Rate",
                field = "mid",
                quote_interval = "1D.0D",
            ),
            lm.MarketDataKeyRule(
                key = "Quote.*.*",
                supplier = "Lusid",
                data_scope = f"{module_scope}{module_code}",
                quote_type = "Price",
                field = "mid",
                quote_interval = "1D.0D",
            )
        ]
    ),
    # Change pricing models
    pricing=lm.PricingContext(
        model_rules=[
            lm.VendorModelRule(
                supplier="Lusid",
                model_name="FlexibleRepoSimplePricer",
                instrument_type="FlexibleRepo"
            ),
            lm.VendorModelRule(
                supplier="Lusid",
                model_name="BondLookupPricer",
                instrument_type="Bond"
            )
        ],
        accrualDefinition="EOD"
    )
)

try:
    recipe_api.upsert_configuration_recipe(
        upsert_recipe_request = lm.UpsertRecipeRequest(
            configuration_recipe = recipe
        )
    )
    print("Success")
except ApiException as e:
    print(e)

Success


In [27]:
# Confirm upsert and show the many options that are automatically set to default values by LUSID.
config_recipe = recipe_api.list_configuration_recipes(filter=f"value.scope eq '{module_scope}' and value.code startswith '{module_code}'")
config_recipe_df = lusid_response_to_data_frame(config_recipe)
display(config_recipe_df.transpose())

,0,1,2,3
value.scope,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials
value.code,FlexRepoOpen-FlexibleRepoSimplePricer,FlexRepo-FlexibleRepoSimplePricer,FlexRepoTest2-FlexibleRepoSimplePricer,FlexRepoTermTest1-FlexibleRepoSimplePricer
value.market.market_rules.0.key,Fx.CurrencyPair.*,Fx.CurrencyPair.*,Fx.CurrencyPair.*,Fx.CurrencyPair.*
value.market.market_rules.0.supplier,Lusid,Lusid,Lusid,Lusid
value.market.market_rules.0.data_scope,FBNTutorialsFlexRepoOpen,FBNTutorialsFlexRepo,FBNTutorialsFlexRepoTest2,FBNTutorialsFlexRepoTermTest1
value.market.market_rules.0.quote_type,Rate,Rate,Rate,Rate
value.market.market_rules.0.var_field,mid,mid,mid,mid
value.market.market_rules.0.quote_interval,1D.0D,1D.0D,1D.0D,1D.0D
value.market.market_rules.0.price_source,,,,
value.market.market_rules.0.source_system,Lusid,Lusid,Lusid,Lusid


## Create a corporate action source

Required if `FlexibleRepoPartialClosureEvent` and/or `FlexibleRepoFullClosureEvent` are to be loaded as manual events. Must be created before portfolios.

In [28]:
corporate_action_sources_api = api_factory.build(la.CorporateActionSourcesApi)

ca_source_definition = lm.CreateCorporateActionSourceRequest(
    scope=module_scope,
    code=module_code,
    display_name=f"{module_scope}/{module_code} CAS",
    instrument_scopes = [f"{module_scope}{module_code}"]
)

try:
    corporate_action_sources_api.create_corporate_action_source(
        create_corporate_action_source_request = ca_source_definition
    )
    print(f"{module_scope}/{module_code} CAS created")
except ApiException as e:
    print(e)

FBNTutorials/FlexRepo CAS created


## Set up GBP transaction portfolios

One for the `Buyer` and one for the `Seller`.

With the portfolio recipe set to the recipe created above to enable instrument events, and the transaction type scope, corporate action source and sub-holding key registered.

In [29]:
ports = ["Buyer-Term", "Seller-Term"]
# ports = ["Buyer-Open", "Seller-Open"]
# ports = ["Buyer-Term", "Seller-Term", "Buyer-Open", "Seller-Open"]

In [30]:
def create_portfolio(name):
    portfolio_request=lm.CreateTransactionPortfolioRequest(
        display_name = f"{name}FlexRepo portfolio",
        code = f"{module_code}-{name}",
        # Set the portfolio currency
        base_currency = "GBP",
        # Must be before first transaction recorded
        created = datetime.strptime("2024-01-01", '%Y-%m-%d').replace(tzinfo=timezone.utc).isoformat(),
        # Attempt to resolve transactions to instruments in the custom scope before falling back to the default scope
        instrument_scopes = [f"{module_scope}{module_code}"],
        # Register SHKs
        sub_holding_keys = [bond_coupon_shk],
        # Register transaction type scope
        transactionTypeScope=f"{module_scope}{module_code}",
        # Register portfolio recipe
        instrumentEventConfiguration=lm.InstrumentEventConfiguration(
            recipeId=lm.ResourceId(
                scope=module_scope,
                code=f"{module_code}-FlexibleRepoSimplePricer"
            )
        ),
        # Register corporate action source
        corporate_action_source_id=lm.ResourceId(
            scope=module_scope,
            code=module_code
        )
    )

    try:
        portfolio_response=transaction_portfolios_api.create_portfolio(
            scope = module_scope,
            create_transaction_portfolio_request = portfolio_request
        )
        print(f"Portfolio with display name '{portfolio_response.display_name}' created effective {str(portfolio_response.created)}")
    except ApiException as e:
        print(e)

In [31]:
for port in ports:
    create_portfolio(port)

Portfolio with display name 'Buyer-TermFlexRepo portfolio' created effective 2024-01-01 00:00:00+00:00
Portfolio with display name 'Seller-TermFlexRepo portfolio' created effective 2024-01-01 00:00:00+00:00


### Confirm portfolio details

In [32]:
def get_port_details(port):
    portfolio_response = transaction_portfolios_api.get_details(scope = module_scope, code = f"{module_code}-{port}")
    portfolio_response_df = lusid_response_to_data_frame(portfolio_response).transpose()
    # Drop some noisy columns
    portfolio_response_df.drop(portfolio_response_df.filter(regex='version|href|staged|links|settlement').columns, axis=1, inplace=True)
    display(portfolio_response_df.transpose())
    
for port in ports:
    get_port_details(port)

,response_values
origin_portfolio_id.scope,FBNTutorials
origin_portfolio_id.code,FlexRepo-Buyer-Term
base_currency,GBP
corporate_action_source_id.scope,FBNTutorials
corporate_action_source_id.code,FlexRepo
sub_holding_keys.0,Transaction/SHKs/BondCollateralCoupons
instrument_scopes.0,FBNTutorialsFlexRepo
accounting_method,Default
amortisation_method,NoAmortisation
transaction_type_scope,FBNTutorialsFlexRepo


,response_values
origin_portfolio_id.scope,FBNTutorials
origin_portfolio_id.code,FlexRepo-Seller-Term
base_currency,GBP
corporate_action_source_id.scope,FBNTutorials
corporate_action_source_id.code,FlexRepo
sub_holding_keys.0,Transaction/SHKs/BondCollateralCoupons
instrument_scopes.0,FBNTutorialsFlexRepo
accounting_method,Default
amortisation_method,NoAmortisation
transaction_type_scope,FBNTutorialsFlexRepo


### Load transactions into portfolios

The `totalConsideration.amount` (ie. cost) needs to be `purchasePrice` to generate correct JE Lines. If this is not explicitly known then it can be calculated as `purchasePrice = collateralValue * (1 - haircut)`.

For a `Buyer` the total consideration must be signed negative, and for a `Seller` it must be signed positive. See https://support.lusid.com/docs/modelling-repurchase-agreements-in-lusid#booking-a-transaction-to-establish-a-position.

In [33]:
tc = myCollateralValue * (1 - myHaircut)
print(tc)

1000000.0


In [34]:
# Create convenience function
def create_transactions(port, txnid, tttype, luid, tradedate, price, quantity, tc, ccy):   
    create_txn_request = {
        "number_one": lm.TransactionRequest(
            transaction_id=txnid,
            type=tttype,
            instrument_identifiers = {"Instrument/default/LusidInstrumentId": luid},
            transaction_date=tradedate,
            settlement_date=tradedate,
            units=quantity,
            transaction_currency = ccy,
            transaction_price = lm.TransactionPrice(
                price = 0,
                type = "Price"
            ),
            total_consideration = lm.CurrencyAndAmount(
                amount = tc,
                currency = ccy,
            )
        )
    }
    
    try:
        create_txn_response = transaction_portfolios_api.batch_upsert_transactions(
            scope = f"{module_scope}",
            code = f"{module_code}-{port}",
            success_mode="Partial",
            request_body = create_txn_request
        )
        print(create_txn_response.failed) if create_txn_response.failed else print("Success")
    except ApiException as e:
        print(e)

In [35]:
# Buy `Bond` collateral in both portfolios before start date of `FlexibleRepo`, at zero cost
for port in ports:
    create_transactions(port, "Txn01", "StockIn", luid_dict["BondCollateral"], "2025-01-01", 100, 3, 0, "GBP")

# Buyer portfolio - total consideration must be negative
create_transactions(ports[0], "Txn02", "EnterRepo", luid_dict[f"{ports[0]}FlexRepo"], "2025-01-01", 0, 1, -tc, "GBP")

# Seller portfolio - total consideration must be positive
create_transactions(ports[1], "Txn02", "EnterRepo", luid_dict[f"{ports[1]}FlexRepo"], "2025-01-01", 0, 1, tc, "GBP")

Success
Success
Success
Success


### Confirm positions and audit output transactions

In [36]:
def get_portfolio_holdings(port, date):      
    if date == "today":
        date = str(datetime.datetime.now().replace(microsecond=0))
    
    try:
        get_holdings_response = transaction_portfolios_api.get_holdings(
            scope = module_scope, 
            code = f"{module_code}-{port}",
            effective_at = datetime.strptime(date, '%Y-%m-%d %H:%M:%S').replace(tzinfo=timezone.utc).isoformat(),
            property_keys=["Instrument/default/Name"]
        )
        get_holdings_response_df = lusid_response_to_data_frame(get_holdings_response)
        get_holdings_response_df.rename(columns = {
           "sub_holding_keys.Transaction/SHKs/BondCollateralCoupons.value.label_value": "SHK",
           "properties.Instrument/default/Name.value.label_value": "instrument"}, inplace = True)        
        # Drop some noisy columns
        get_holdings_response_df.drop(get_holdings_response_df.filter(regex='properties|sub_holding_keys').columns, axis=1, inplace=True)
        display(get_holdings_response_df)
    except ApiException as e:
        print(e)

In [37]:
def get_output_transactions(port, start, end):
    try:
        output_transactions_response = transaction_portfolios_api.build_transactions(
            scope = module_scope, 
            code = f"{module_code}-{port}",
            transaction_query_parameters = lm.TransactionQueryParameters(
                start_date = datetime.strptime(start, '%Y-%m-%d').replace(tzinfo=timezone.utc).isoformat(),
                end_date = datetime.strptime(end, '%Y-%m-%d').replace(tzinfo=timezone.utc).isoformat()
            )
        )
        output_transactions_response_df = lusid_response_to_data_frame(output_transactions_response)
        display(output_transactions_response_df.transpose())
    except ApiException as e:
        print(e)

In [38]:
for port in ports:
    print(f"\n{port}")
    get_portfolio_holdings(port, "2025-01-01 00:00:00")   # Transaction date
    get_portfolio_holdings(port, "2025-01-02 00:00:00")   # Instrument start date
    get_portfolio_holdings(port, "2025-03-31 00:00:00")   # Instrument maturity date


Buyer-Term


,instrument_scope,instrument_uid,SHK,instrument,holding_type,units,settled_units,cost.amount,cost.currency,cost_portfolio_ccy.amount,cost_portfolio_ccy.currency,currency,holding_type_name,holding_id,notional_cost.amount,notional_cost.currency,amortised_cost.amount,amortised_cost.currency,amortised_cost_portfolio_ccy.amount,amortised_cost_portfolio_ccy.currency,variation_margin.amount,variation_margin.currency,variation_margin_portfolio_ccy.amount,variation_margin_portfolio_ccy.currency,settlement_schedule,unsettled_units,overdue_units
0,FBNTutorialsFlexRepo,LUID_00003H1O,<Not Classified>,BondCollateral,P,3.00,3.00,0.00,GBP,0.00,GBP,GBP,Position,81088532,0.00,GBP,0.00,GBP,0.00,GBP,0.00,GBP,0.00,GBP,[],0.00,0.00
1,FBNTutorialsFlexRepo,LUID_00003H1P,<Not Classified>,Buyer-TermFlexRepo,P,1.00,1.00,"1,000,000.00",GBP,"1,000,000.00",GBP,GBP,Position,81088533,0.00,GBP,"1,000,000.00",GBP,"1,000,000.00",GBP,0.00,GBP,0.00,GBP,[],0.00,0.00
2,default,CCY_GBP,<Not Classified>,GBP,B,"-1,000,000.00","-1,000,000.00","-1,000,000.00",GBP,"-1,000,000.00",GBP,GBP,Balance,81088534,0.00,GBP,"-1,000,000.00",GBP,"-1,000,000.00",GBP,0.00,GBP,0.00,GBP,[],0.00,0.00


,instrument_scope,instrument_uid,SHK,instrument,holding_type,units,settled_units,cost.amount,cost.currency,cost_portfolio_ccy.amount,cost_portfolio_ccy.currency,currency,holding_type_name,holding_id,notional_cost.amount,notional_cost.currency,amortised_cost.amount,amortised_cost.currency,amortised_cost_portfolio_ccy.amount,amortised_cost_portfolio_ccy.currency,variation_margin.amount,variation_margin.currency,variation_margin_portfolio_ccy.amount,variation_margin_portfolio_ccy.currency,settlement_schedule,unsettled_units,overdue_units
0,FBNTutorialsFlexRepo,LUID_00003H1O,<Not Classified>,BondCollateral,P,4.00,4.00,0.00,GBP,0.00,GBP,GBP,Position,81088532,0.00,GBP,0.00,GBP,0.00,GBP,0.00,GBP,0.00,GBP,[],0.00,0.00
1,FBNTutorialsFlexRepo,LUID_00003H1P,<Not Classified>,Buyer-TermFlexRepo,P,1.00,1.00,"1,000,000.00",GBP,"1,000,000.00",GBP,GBP,Position,81088533,0.00,GBP,"1,000,000.00",GBP,"1,000,000.00",GBP,0.00,GBP,0.00,GBP,[],0.00,0.00
2,default,CCY_GBP,<Not Classified>,GBP,B,"-1,000,000.00","-1,000,000.00","-1,000,000.00",GBP,"-1,000,000.00",GBP,GBP,Balance,81088534,0.00,GBP,"-1,000,000.00",GBP,"-1,000,000.00",GBP,0.00,GBP,0.00,GBP,[],0.00,0.00


,instrument_scope,instrument_uid,SHK,instrument,holding_type,units,settled_units,cost.amount,cost.currency,cost_portfolio_ccy.amount,cost_portfolio_ccy.currency,currency,holding_type_name,holding_id,notional_cost.amount,notional_cost.currency,amortised_cost.amount,amortised_cost.currency,amortised_cost_portfolio_ccy.amount,amortised_cost_portfolio_ccy.currency,variation_margin.amount,variation_margin.currency,variation_margin_portfolio_ccy.amount,variation_margin_portfolio_ccy.currency,settlement_schedule,unsettled_units,overdue_units
0,FBNTutorialsFlexRepo,LUID_00003H1O,<Not Classified>,BondCollateral,P,3.00,3.00,0.00,GBP,0.00,GBP,GBP,Position,81088532,0.00,GBP,0.00,GBP,0.00,GBP,0.00,GBP,0.00,GBP,[],0.00,0.00
1,default,CCY_GBP,<Not Classified>,GBP,B,"12,054.79","12,054.79","12,054.79",GBP,"12,054.79",GBP,GBP,Balance,81088534,0.00,GBP,"12,054.79",GBP,"12,054.79",GBP,0.00,GBP,0.00,GBP,[],0.00,0.00
2,default,CCY_GBP,BondCollateralCoupons,GBP,B,187.50,187.50,187.50,GBP,187.50,GBP,GBP,Balance,81088535,0.00,GBP,187.50,GBP,187.50,GBP,0.00,GBP,0.00,GBP,[],0.00,0.00



Seller-Term


,instrument_scope,instrument_uid,SHK,instrument,holding_type,units,settled_units,cost.amount,cost.currency,cost_portfolio_ccy.amount,cost_portfolio_ccy.currency,currency,holding_type_name,holding_id,notional_cost.amount,notional_cost.currency,amortised_cost.amount,amortised_cost.currency,amortised_cost_portfolio_ccy.amount,amortised_cost_portfolio_ccy.currency,variation_margin.amount,variation_margin.currency,variation_margin_portfolio_ccy.amount,variation_margin_portfolio_ccy.currency,settlement_schedule,unsettled_units,overdue_units
0,FBNTutorialsFlexRepo,LUID_00003H1O,<Not Classified>,BondCollateral,P,3.00,3.00,0.00,GBP,0.00,GBP,GBP,Position,81088536,0.00,GBP,0.00,GBP,0.00,GBP,0.00,GBP,0.00,GBP,[],0.00,0.00
1,FBNTutorialsFlexRepo,LUID_00003H1Q,<Not Classified>,Seller-TermFlexRepo,P,1.00,1.00,"-1,000,000.00",GBP,"-1,000,000.00",GBP,GBP,Position,81088537,0.00,GBP,"-1,000,000.00",GBP,"-1,000,000.00",GBP,0.00,GBP,0.00,GBP,[],0.00,0.00
2,default,CCY_GBP,<Not Classified>,GBP,B,"1,000,000.00","1,000,000.00","1,000,000.00",GBP,"1,000,000.00",GBP,GBP,Balance,81088538,0.00,GBP,"1,000,000.00",GBP,"1,000,000.00",GBP,0.00,GBP,0.00,GBP,[],0.00,0.00


,instrument_scope,instrument_uid,SHK,instrument,holding_type,units,settled_units,cost.amount,cost.currency,cost_portfolio_ccy.amount,cost_portfolio_ccy.currency,currency,holding_type_name,holding_id,notional_cost.amount,notional_cost.currency,amortised_cost.amount,amortised_cost.currency,amortised_cost_portfolio_ccy.amount,amortised_cost_portfolio_ccy.currency,variation_margin.amount,variation_margin.currency,variation_margin_portfolio_ccy.amount,variation_margin_portfolio_ccy.currency,settlement_schedule,unsettled_units,overdue_units
0,FBNTutorialsFlexRepo,LUID_00003H1O,<Not Classified>,BondCollateral,P,2.00,2.00,0.00,GBP,0.00,GBP,GBP,Position,81088536,0.00,GBP,0.00,GBP,0.00,GBP,0.00,GBP,0.00,GBP,[],0.00,0.00
1,FBNTutorialsFlexRepo,LUID_00003H1Q,<Not Classified>,Seller-TermFlexRepo,P,1.00,1.00,"-1,000,000.00",GBP,"-1,000,000.00",GBP,GBP,Position,81088537,0.00,GBP,"-1,000,000.00",GBP,"-1,000,000.00",GBP,0.00,GBP,0.00,GBP,[],0.00,0.00
2,default,CCY_GBP,<Not Classified>,GBP,B,"1,000,000.00","1,000,000.00","1,000,000.00",GBP,"1,000,000.00",GBP,GBP,Balance,81088538,0.00,GBP,"1,000,000.00",GBP,"1,000,000.00",GBP,0.00,GBP,0.00,GBP,[],0.00,0.00


,instrument_scope,instrument_uid,SHK,instrument,holding_type,units,settled_units,cost.amount,cost.currency,cost_portfolio_ccy.amount,cost_portfolio_ccy.currency,currency,holding_type_name,holding_id,notional_cost.amount,notional_cost.currency,amortised_cost.amount,amortised_cost.currency,amortised_cost_portfolio_ccy.amount,amortised_cost_portfolio_ccy.currency,variation_margin.amount,variation_margin.currency,variation_margin_portfolio_ccy.amount,variation_margin_portfolio_ccy.currency,settlement_schedule,unsettled_units,overdue_units
0,FBNTutorialsFlexRepo,LUID_00003H1O,<Not Classified>,BondCollateral,P,3.00,3.00,0.00,GBP,0.00,GBP,GBP,Position,81088536,0.00,GBP,0.00,GBP,0.00,GBP,0.00,GBP,0.00,GBP,[],0.00,0.00
1,default,CCY_GBP,<Not Classified>,GBP,B,"-12,054.79","-12,054.79","-12,054.79",GBP,"-12,054.79",GBP,GBP,Balance,81088538,0.00,GBP,"-12,054.79",GBP,"-12,054.79",GBP,0.00,GBP,0.00,GBP,[],0.00,0.00
2,default,CCY_GBP,BondCollateralCoupons,GBP,B,187.50,187.50,187.50,GBP,187.50,GBP,GBP,Balance,81088539,0.00,GBP,187.50,GBP,187.50,GBP,0.00,GBP,0.00,GBP,[],0.00,0.00


In [39]:
for port in ports:
    print(f"\n{port}")
    get_output_transactions(port, "2025-01-01", "2025-06-01")


Buyer-Term


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
transaction_id,Txn01,Txn02,LUID_00003H1P_FlexibleRepoCollateralEvent_2025...,LUID_00003H1O_BondCouponEvent_20250128-81088532,LUID_00003H1O_BondCouponEvent_20250128-Manufac...,LUID_00003H1O_BondCouponEvent_20250228-81088532,LUID_00003H1O_BondCouponEvent_20250228-Manufac...,LUID_00003H1O_BondCouponEvent_20250328-81088532,LUID_00003H1O_BondCouponEvent_20250328-Manufac...,LUID_00003H1P_FlexibleRepoCashFlowEvent_2025-0...,LUID_00003H1P_FlexibleRepoCollateralEvent_2025...,LUID_00003H1P_FlexibleRepoInterestPaymentEvent...,LUID_00003H1P_MaturityEvent_20250331-81088533,LUID_00003H1O_BondCouponEvent_20250428-81088532,LUID_00003H1O_BondCouponEvent_20250528-81088532
type,StockIn,EnterRepo,FlexibleRepoCollateralTransfer,BondCoupon,BondCoupon,BondCoupon,BondCoupon,BondCoupon,BondCoupon,FlexibleRepoCashTransfer,FlexibleRepoCollateralTransfer,FlexibleRepoInterestPayment,Maturity,BondCoupon,BondCoupon
description,Transfer in,Open the contract,Transaction type for transactions automaticall...,Type for bond coupon event,Type for bond coupon event,Type for bond coupon event,Type for bond coupon event,Type for bond coupon event,Type for bond coupon event,Transaction type for transactions automaticall...,Transaction type for transactions automaticall...,Transaction type for transactions automaticall...,Type for maturity event for various instruments,Type for bond coupon event,Type for bond coupon event
instrument_identifiers.Instrument/default/LusidInstrumentId,LUID_00003H1O,LUID_00003H1P,LUID_00003H1O,LUID_00003H1O,LUID_00003H1P,LUID_00003H1O,LUID_00003H1P,LUID_00003H1O,LUID_00003H1P,LUID_00003H1P,LUID_00003H1O,LUID_00003H1P,LUID_00003H1P,LUID_00003H1O,LUID_00003H1O
instrument_scope,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo
instrument_uid,LUID_00003H1O,LUID_00003H1P,LUID_00003H1O,LUID_00003H1O,LUID_00003H1P,LUID_00003H1O,LUID_00003H1P,LUID_00003H1O,LUID_00003H1P,LUID_00003H1P,LUID_00003H1O,LUID_00003H1P,LUID_00003H1P,LUID_00003H1O,LUID_00003H1O
transaction_date,2025-01-01 00:00:00+00:00,2025-01-01 00:00:00+00:00,2025-01-02 00:00:00+00:00,2025-01-28 00:00:00+00:00,2025-01-28 00:00:00+00:00,2025-02-28 00:00:00+00:00,2025-02-28 00:00:00+00:00,2025-03-28 00:00:00+00:00,2025-03-28 00:00:00+00:00,2025-03-31 00:00:00+00:00,2025-03-31 00:00:00+00:00,2025-03-31 00:00:00+00:00,2025-03-31 00:00:00+00:00,2025-04-28 00:00:00+00:00,2025-05-28 00:00:00+00:00
settlement_date,2025-01-01 00:00:00+00:00,2025-01-01 00:00:00+00:00,2025-01-02 00:00:00+00:00,2025-01-28 00:00:00+00:00,2025-01-28 00:00:00+00:00,2025-02-28 00:00:00+00:00,2025-02-28 00:00:00+00:00,2025-03-28 00:00:00+00:00,2025-03-28 00:00:00+00:00,2025-03-31 00:00:00+00:00,2025-03-31 00:00:00+00:00,2025-03-31 00:00:00+00:00,2025-03-31 00:00:00+00:00,2025-04-28 00:00:00+00:00,2025-05-28 00:00:00+00:00
units,3.00,1.00,1.00,4.00,1.00,4.00,1.00,4.00,1.00,1.00,-1.00,1.00,1.00,3.00,3.00
transaction_amount,0.00,"1,000,000.00",0.00,83.33,20.83,83.33,20.83,83.33,20.83,"1,000,000.00",0.00,"12,054.79",0.00,62.50,62.50



Seller-Term


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
transaction_id,Txn01,Txn02,LUID_00003H1Q_FlexibleRepoCollateralEvent_2025...,LUID_00003H1O_BondCouponEvent_20250128-81088536,LUID_00003H1O_BondCouponEvent_20250128-Manufac...,LUID_00003H1O_BondCouponEvent_20250228-81088536,LUID_00003H1O_BondCouponEvent_20250228-Manufac...,LUID_00003H1O_BondCouponEvent_20250328-81088536,LUID_00003H1O_BondCouponEvent_20250328-Manufac...,LUID_00003H1Q_FlexibleRepoCashFlowEvent_2025-0...,LUID_00003H1Q_FlexibleRepoCollateralEvent_2025...,LUID_00003H1Q_FlexibleRepoInterestPaymentEvent...,LUID_00003H1Q_MaturityEvent_20250331-81088537,LUID_00003H1O_BondCouponEvent_20250428-81088536,LUID_00003H1O_BondCouponEvent_20250528-81088536
type,StockIn,EnterRepo,FlexibleRepoCollateralTransfer,BondCoupon,BondCoupon,BondCoupon,BondCoupon,BondCoupon,BondCoupon,FlexibleRepoCashTransfer,FlexibleRepoCollateralTransfer,FlexibleRepoInterestPayment,Maturity,BondCoupon,BondCoupon
description,Transfer in,Open the contract,Transaction type for transactions automaticall...,Type for bond coupon event,Type for bond coupon event,Type for bond coupon event,Type for bond coupon event,Type for bond coupon event,Type for bond coupon event,Transaction type for transactions automaticall...,Transaction type for transactions automaticall...,Transaction type for transactions automaticall...,Type for maturity event for various instruments,Type for bond coupon event,Type for bond coupon event
instrument_identifiers.Instrument/default/LusidInstrumentId,LUID_00003H1O,LUID_00003H1Q,LUID_00003H1O,LUID_00003H1O,LUID_00003H1Q,LUID_00003H1O,LUID_00003H1Q,LUID_00003H1O,LUID_00003H1Q,LUID_00003H1Q,LUID_00003H1O,LUID_00003H1Q,LUID_00003H1Q,LUID_00003H1O,LUID_00003H1O
instrument_scope,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo
instrument_uid,LUID_00003H1O,LUID_00003H1Q,LUID_00003H1O,LUID_00003H1O,LUID_00003H1Q,LUID_00003H1O,LUID_00003H1Q,LUID_00003H1O,LUID_00003H1Q,LUID_00003H1Q,LUID_00003H1O,LUID_00003H1Q,LUID_00003H1Q,LUID_00003H1O,LUID_00003H1O
transaction_date,2025-01-01 00:00:00+00:00,2025-01-01 00:00:00+00:00,2025-01-02 00:00:00+00:00,2025-01-28 00:00:00+00:00,2025-01-28 00:00:00+00:00,2025-02-28 00:00:00+00:00,2025-02-28 00:00:00+00:00,2025-03-28 00:00:00+00:00,2025-03-28 00:00:00+00:00,2025-03-31 00:00:00+00:00,2025-03-31 00:00:00+00:00,2025-03-31 00:00:00+00:00,2025-03-31 00:00:00+00:00,2025-04-28 00:00:00+00:00,2025-05-28 00:00:00+00:00
settlement_date,2025-01-01 00:00:00+00:00,2025-01-01 00:00:00+00:00,2025-01-02 00:00:00+00:00,2025-01-28 00:00:00+00:00,2025-01-28 00:00:00+00:00,2025-02-28 00:00:00+00:00,2025-02-28 00:00:00+00:00,2025-03-28 00:00:00+00:00,2025-03-28 00:00:00+00:00,2025-03-31 00:00:00+00:00,2025-03-31 00:00:00+00:00,2025-03-31 00:00:00+00:00,2025-03-31 00:00:00+00:00,2025-04-28 00:00:00+00:00,2025-05-28 00:00:00+00:00
units,3.00,1.00,-1.00,2.00,1.00,2.00,1.00,2.00,1.00,1.00,1.00,1.00,1.00,3.00,3.00
transaction_amount,0.00,"1,000,000.00",0.00,41.67,20.83,41.67,20.83,41.67,20.83,"1,000,000.00",0.00,"12,054.79",0.00,62.50,62.50


## Valuation

See https://support.lusid.com/docs/modelling-repurchase-agreements-in-lusid#valuing-your-position.

### Load market data

Using `FlexibleRepoSimplePricer`, there is no need to load market prices for `FlexibleRepo` instruments. Only market prices for the collateral instrument (in this case, `Bond`) required.

In [40]:
def load_quotes(luid, price, date, ccy, scale):
    if date == "today":
        date = str(datetime.datetime.now().replace(microsecond=0))

    quotes = {
        # Each quote must be upserted with an ephemeral key (uuid in this case), to track errors in the response
        str(uuid.uuid4()): lm.UpsertQuoteRequest(
            quote_id = lm.QuoteId(
                quote_series_id = lm.QuoteSeriesId(
                    # Must be one of the valid financial data vendor 'provider' values
                    provider = "Lusid",
                    instrument_id_type = "LusidInstrumentId",
                    instrument_id = luid,
                    quote_type = "Price",
                    # Case sensitive: the field value must match that of the equivalent recipe field exactly
                    field = "mid",
                ),
                effective_at = datetime.strptime(date, '%Y-%m-%d %H:%M:%S').replace(tzinfo=timezone.utc).isoformat(),
            ),
            metric_value = lm.MetricValue(value = price, unit = ccy),
            scale_factor = scale,
        )
    }

    try:
        upsert_quotes_response = quotes_api.upsert_quotes(scope = f"{module_scope}{module_code}", request_body = quotes)    
        if upsert_quotes_response.failed == {}:
            print(f"Price for {date} successfully loaded into LUSID.")
        else:
            print(f"Some failures occurred. {len(upsert_quotes_response.failed)} prices did not get loaded into LUSID.")
    except ApiException as e:
        print(e)

In [41]:
load_quotes(luid_dict["BondCollateral"], 100, "2025-01-01 00:00:00", "GBP", 100) # Transaction date (required to calculate YTD P&L)
load_quotes(luid_dict["BondCollateral"], 101, "2025-02-14 00:00:00", "GBP", 100) # Random valuation date

Price for 2025-01-01 00:00:00 successfully loaded into LUSID.
Price for 2025-02-14 00:00:00 successfully loaded into LUSID.


In [42]:
def list_quotes():
    try:
        quotes_response = quotes_api.list_quotes_for_scope(f"{module_scope}{module_code}")
        quotes_response_df = lusid_response_to_data_frame(quotes_response)
        display(quotes_response_df)        
    except ApiException as e:
        print(e)

list_quotes()

,quote_id.quote_series_id.provider,quote_id.quote_series_id.instrument_id,quote_id.quote_series_id.instrument_id_type,quote_id.quote_series_id.quote_type,quote_id.quote_series_id.var_field,quote_id.quote_series_id.entity_unique_id,quote_id.effective_at,metric_value.value,metric_value.unit,lineage,cut_label,uploaded_by,as_at,scale_factor
0,Lusid,LUID_00003H1O,LusidInstrumentId,Price,mid,e1c303f7-563f-4d08-81ca-d0946dc56637,2025-02-14T00:00:00.0000000+00:00,101.00,GBP,,,00u91lo2d7X42sdse2p7,2026-06-10 11:45:58.042026+00:00,100.00
1,Lusid,LUID_00003H1O,LusidInstrumentId,Price,mid,e1c303f7-563f-4d08-81ca-d0946dc56637,2025-01-01T00:00:00.0000000+00:00,100.00,GBP,,,00u91lo2d7X42sdse2p7,2026-06-10 11:45:57.855914+00:00,100.00


In [43]:
def value_instruments(port, date, pnl_window):
    if date == "today":
        date = str(datetime.now().replace(microsecond=0))

    valuation_request = lm.ValuationRequest(
        # Choose recipe to use
        recipe_id = lm.ResourceId(scope = module_scope, code = f"{module_code}-FlexibleRepoSimplePricer"),
        # Specify metrics (also known as queryable keys) to report useful information
        metrics = [
            lm.AggregateSpec(key="Instrument/InstrumentCategory", op="Value"),
            lm.AggregateSpec(key="Valuation/Model/Name", op="Value"),
            lm.AggregateSpec(key="Instrument/default/LusidInstrumentId", op="Value"),
            lm.AggregateSpec(key="Instrument/default/Name", op="Value"),
            lm.AggregateSpec(key="Valuation/EffectiveAt", op="Value"),
            lm.AggregateSpec(key="Holding/default/Units", op="Value"),
            lm.AggregateSpec(key="Quotes/PriceOrFXRate", op="Value"),
            lm.AggregateSpec(key="Holding/Cost/Dom", op="Value"),
            lm.AggregateSpec(key="Valuation/CleanPV", op="Value"),
            lm.AggregateSpec(key="Valuation/PV", op="Value"),
            lm.AggregateSpec(key="Valuation/Accrued", op="Value"),
            lm.AggregateSpec(key="Valuation/Exposure", op="Value"),
            lm.AggregateSpec(key="Valuation/CurrentNotional", op="Value"), 
            lm.AggregateSpec(key="ProfitAndLoss/Total", op="Value", options={"Window": f"{pnl_window}"}),      
            lm.AggregateSpec(key="ProfitAndLoss/Total/Market", op="Value", options={"Window": f"{pnl_window}"}),
            lm.AggregateSpec(key="ProfitAndLoss/Realised/Market", op="Value", options={"Window": f"{pnl_window}"}),       
            lm.AggregateSpec(key="ProfitAndLoss/Unrealised/Market", op="Value", options={"Window": f"{pnl_window}"}),       
            lm.AggregateSpec(key="ProfitAndLoss/Total/Other", op="Value", options={"Window": f"{pnl_window}"}),
            lm.AggregateSpec(key="Aggregation/Errors", op="Value"), 
        ],
        # Identify portfolio to value
        portfolio_entity_ids = [lm.PortfolioEntityId(scope = module_scope, code = f"{module_code}-{port}")],
        valuation_schedule = lm.ValuationSchedule(effective_at = date),

    )

    try:
        # Get portfolio valuation
        val_response = aggregation_api.get_valuation(valuation_request = valuation_request)
        val_response_df = pd.json_normalize(val_response.to_dict()["data"], sep='.')
        # Rename columns
        val_response_df.rename(
            columns = {
                "Instrument/InstrumentCategory": "Category",
                "Valuation/Model/Name": "Model",
                "Instrument/default/LusidInstrumentId": "LUID",
                "Instrument/default/Name": "Name",
                "Valuation/EffectiveAt": "Date",
                "Holding/default/Units": "Units",
                "Quotes/PriceOrFXRate": "Price",
                "Quotes/ScaleFactor": "Quote Scale Factor",
                "Holding/Cost/Dom": "Local Cost",
                "Valuation/CleanPV": "Local Clean PV",
                "Valuation/PV": "Local PV",
                "Valuation/Accrued": "Local Accrued Interest",
                "Valuation/Exposure": "Exposure",
                "Valuation/CurrentNotional": "Notional",            
                f"ProfitAndLoss/Total(Window=\"{pnl_window}\")": "Total P&L",
                f"ProfitAndLoss/Total/Market(Window=\"{pnl_window}\")": "Total/Market P&L",
                f"ProfitAndLoss/Realised/Market(Window=\"{pnl_window}\")": "Realised/Market P&L",
                f"ProfitAndLoss/Unrealised/Market(Window=\"{pnl_window}\")": "Unrealised/Market P&L",
                f"ProfitAndLoss/Total/Other(Window=\"{pnl_window}\")": "Total/Other P&L",
                "Aggregation/Errors": "Errors"
            },
            inplace = True,
        )
    #     #val_each_instrument_df["Date"] = pd.to_datetime(val_each_instrument_df["Date"]).dt.date
        display(val_response_df)
    except ApiException as e:
        print(e)

In [44]:
for port in ports:
    print(port)
    value_instruments(port, "2025-02-15T17:00:00Z", "YTD") # Random valuation date

Buyer-Term


,Category,Model,LUID,Name,Date,Units,Price,Local Cost,Local Clean PV,Local PV,Local Accrued Interest,Exposure,Notional,Total P&L,Total/Market P&L,Realised/Market P&L,Unrealised/Market P&L,Total/Other P&L,Errors
0,Bond,BondLookupPricer,LUID_00003H1O,BondCollateral,2025-02-15T17:00:00.0000000+00:00,4.00,101.00,0.00,"4,040.00","4,092.05",52.05,"4,092.05","1,000.00","4,175.38","4,040.00",0.00,"4,040.00",135.38,[]
1,FlexibleRepo,FlexibleRepoSimplePricer,LUID_00003H1P,Buyer-TermFlexRepo,2025-02-15T17:00:00.0000000+00:00,1.00,1.00,"1,000,000.00","1,000,000.00","1,006,164.38","6,164.38","1,100,000.00","1,000,000.00","6,185.21",-0.00,0.00,-0.00,"6,185.21",[]
2,Cash,ConstantTimeValueOfMoney,CCY_GBP,GBP,2025-02-15T17:00:00.0000000+00:00,"-1,000,000.00",1.00,"-1,000,000.00","-1,000,000.00","-1,000,000.00",0.00,"-1,000,000.00",1.00,0.00,0.00,0.00,0.00,0.00,[]
3,Cash,ConstantTimeValueOfMoney,CCY_GBP,GBP,2025-02-15T17:00:00.0000000+00:00,62.50,1.00,62.50,62.50,62.50,0.00,62.50,1.00,0.00,0.00,0.00,0.00,0.00,[]


Seller-Term


,Category,Model,LUID,Name,Date,Units,Price,Local Cost,Local Clean PV,Local PV,Local Accrued Interest,Exposure,Notional,Total P&L,Total/Market P&L,Realised/Market P&L,Unrealised/Market P&L,Total/Other P&L,Errors
0,Bond,BondLookupPricer,LUID_00003H1O,BondCollateral,2025-02-15T17:00:00.0000000+00:00,2.00,101.00,0.00,"2,020.00","2,046.03",26.03,"2,046.03","1,000.00","2,087.70","2,020.00",0.00,"2,020.00",67.70,[]
1,FlexibleRepo,FlexibleRepoSimplePricer,LUID_00003H1Q,Seller-TermFlexRepo,2025-02-15T17:00:00.0000000+00:00,1.00,1.00,"-1,000,000.00","-1,000,000.00","-1,006,164.38","-6,164.38","-1,100,000.00","-1,000,000.00","-6,143.55",0.00,0.00,0.00,"-6,143.55",[]
2,Cash,ConstantTimeValueOfMoney,CCY_GBP,GBP,2025-02-15T17:00:00.0000000+00:00,"1,000,000.00",1.00,"1,000,000.00","1,000,000.00","1,000,000.00",0.00,"1,000,000.00",1.00,0.00,0.00,0.00,0.00,0.00,[]
3,Cash,ConstantTimeValueOfMoney,CCY_GBP,GBP,2025-02-15T17:00:00.0000000+00:00,62.50,1.00,62.50,62.50,62.50,0.00,62.50,1.00,0.00,0.00,0.00,0.00,0.00,[]


## Check instrument events that generated output transactions

In [45]:
def query_instr_events(port):
    
    query_id_request = lm.QueryApplicableInstrumentEventsRequest(
        window_start = datetime.strptime("2015-01-01", '%Y-%m-%d').replace(tzinfo=timezone.utc).isoformat(),
        window_end = datetime.strptime("2025-05-31", '%Y-%m-%d').replace(tzinfo=timezone.utc).isoformat(),
        effective_at = datetime.strptime("2025-03-31", '%Y-%m-%d').replace(tzinfo=timezone.utc).isoformat(),
        portfolio_entity_ids = [
            lm.PortfolioEntityId(
                scope = module_scope,
                code = f"{module_code}-{port}",
            )
        ],
        forecasting_recipe_id = lm.ResourceId(
            scope = module_scope,
            code = f"{module_code}-FlexibleRepoSimplePricer"
        )
    )
    
    try:
        query_id_response = instrument_events_api.query_applicable_instrument_events(
            query_applicable_instrument_events_request = query_id_request,
            limit = 200
        )
        display(lusid_response_to_data_frame(query_id_response).transpose())
    except ApiException as e:
        print(e)

In [46]:
for port in ports:
    print(port)
    query_instr_events(port)

Buyer-Term


,0,1,2,3,4,5,6,7,8,9,10,11,12
portfolio_id.scope,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials
portfolio_id.code,FlexRepo-Buyer-Term,FlexRepo-Buyer-Term,FlexRepo-Buyer-Term,FlexRepo-Buyer-Term,FlexRepo-Buyer-Term,FlexRepo-Buyer-Term,FlexRepo-Buyer-Term,FlexRepo-Buyer-Term,FlexRepo-Buyer-Term,FlexRepo-Buyer-Term,FlexRepo-Buyer-Term,FlexRepo-Buyer-Term,FlexRepo-Buyer-Term
holding_id,81088533,81088532,81088533,81088532,81088533,81088532,81088533,81088533,81088533,81088533,81088533,81088532,81088532
lusid_instrument_id,LUID_00003H1P,LUID_00003H1O,LUID_00003H1P,LUID_00003H1O,LUID_00003H1P,LUID_00003H1O,LUID_00003H1P,LUID_00003H1P,LUID_00003H1P,LUID_00003H1P,LUID_00003H1P,LUID_00003H1O,LUID_00003H1O
instrument_scope,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo
instrument_type,FlexibleRepo,Bond,FlexibleRepo,Bond,FlexibleRepo,Bond,FlexibleRepo,FlexibleRepo,FlexibleRepo,FlexibleRepo,FlexibleRepo,Bond,Bond
instrument_event_type,FlexibleRepoCollateralEvent,BondCouponEvent,BondCouponEvent,BondCouponEvent,BondCouponEvent,BondCouponEvent,BondCouponEvent,FlexibleRepoCashFlowEvent,FlexibleRepoCollateralEvent,FlexibleRepoInterestPaymentEvent,MaturityEvent,BondCouponEvent,BondCouponEvent
instrument_event_id,LUID_00003H1P_FlexibleRepoCollateralEvent_2025...,LUID_00003H1O_BondCouponEvent_20250128,LUID_00003H1O_BondCouponEvent_20250128-Manufac...,LUID_00003H1O_BondCouponEvent_20250228,LUID_00003H1O_BondCouponEvent_20250228-Manufac...,LUID_00003H1O_BondCouponEvent_20250328,LUID_00003H1O_BondCouponEvent_20250328-Manufac...,LUID_00003H1P_FlexibleRepoCashFlowEvent_2025-0...,LUID_00003H1P_FlexibleRepoCollateralEvent_2025...,LUID_00003H1P_FlexibleRepoInterestPaymentEvent...,LUID_00003H1P_MaturityEvent_20250331,LUID_00003H1O_BondCouponEvent_20250428,LUID_00003H1O_BondCouponEvent_20250528
generated_event.instrument_event_id,LUID_00003H1P_FlexibleRepoCollateralEvent_2025...,LUID_00003H1O_BondCouponEvent_20250128,LUID_00003H1O_BondCouponEvent_20250128-Manufac...,LUID_00003H1O_BondCouponEvent_20250228,LUID_00003H1O_BondCouponEvent_20250228-Manufac...,LUID_00003H1O_BondCouponEvent_20250328,LUID_00003H1O_BondCouponEvent_20250328-Manufac...,LUID_00003H1P_FlexibleRepoCashFlowEvent_2025-0...,LUID_00003H1P_FlexibleRepoCollateralEvent_2025...,LUID_00003H1P_FlexibleRepoInterestPaymentEvent...,LUID_00003H1P_MaturityEvent_20250331,LUID_00003H1O_BondCouponEvent_20250428,LUID_00003H1O_BondCouponEvent_20250528
generated_event.instrument_identifiers.Instrument/default/ClientInternal,Buyer-TermFlexRepo,BondCollateral,Buyer-TermFlexRepo,BondCollateral,Buyer-TermFlexRepo,BondCollateral,Buyer-TermFlexRepo,Buyer-TermFlexRepo,Buyer-TermFlexRepo,Buyer-TermFlexRepo,Buyer-TermFlexRepo,BondCollateral,BondCollateral


Seller-Term


,0,1,2,3,4,5,6,7,8,9,10,11,12
portfolio_id.scope,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials,FBNTutorials
portfolio_id.code,FlexRepo-Seller-Term,FlexRepo-Seller-Term,FlexRepo-Seller-Term,FlexRepo-Seller-Term,FlexRepo-Seller-Term,FlexRepo-Seller-Term,FlexRepo-Seller-Term,FlexRepo-Seller-Term,FlexRepo-Seller-Term,FlexRepo-Seller-Term,FlexRepo-Seller-Term,FlexRepo-Seller-Term,FlexRepo-Seller-Term
holding_id,81088537,81088536,81088537,81088536,81088537,81088536,81088537,81088537,81088537,81088537,81088537,81088536,81088536
lusid_instrument_id,LUID_00003H1Q,LUID_00003H1O,LUID_00003H1Q,LUID_00003H1O,LUID_00003H1Q,LUID_00003H1O,LUID_00003H1Q,LUID_00003H1Q,LUID_00003H1Q,LUID_00003H1Q,LUID_00003H1Q,LUID_00003H1O,LUID_00003H1O
instrument_scope,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo,FBNTutorialsFlexRepo
instrument_type,FlexibleRepo,Bond,FlexibleRepo,Bond,FlexibleRepo,Bond,FlexibleRepo,FlexibleRepo,FlexibleRepo,FlexibleRepo,FlexibleRepo,Bond,Bond
instrument_event_type,FlexibleRepoCollateralEvent,BondCouponEvent,BondCouponEvent,BondCouponEvent,BondCouponEvent,BondCouponEvent,BondCouponEvent,FlexibleRepoCashFlowEvent,FlexibleRepoCollateralEvent,FlexibleRepoInterestPaymentEvent,MaturityEvent,BondCouponEvent,BondCouponEvent
instrument_event_id,LUID_00003H1Q_FlexibleRepoCollateralEvent_2025...,LUID_00003H1O_BondCouponEvent_20250128,LUID_00003H1O_BondCouponEvent_20250128-Manufac...,LUID_00003H1O_BondCouponEvent_20250228,LUID_00003H1O_BondCouponEvent_20250228-Manufac...,LUID_00003H1O_BondCouponEvent_20250328,LUID_00003H1O_BondCouponEvent_20250328-Manufac...,LUID_00003H1Q_FlexibleRepoCashFlowEvent_2025-0...,LUID_00003H1Q_FlexibleRepoCollateralEvent_2025...,LUID_00003H1Q_FlexibleRepoInterestPaymentEvent...,LUID_00003H1Q_MaturityEvent_20250331,LUID_00003H1O_BondCouponEvent_20250428,LUID_00003H1O_BondCouponEvent_20250528
generated_event.instrument_event_id,LUID_00003H1Q_FlexibleRepoCollateralEvent_2025...,LUID_00003H1O_BondCouponEvent_20250128,LUID_00003H1O_BondCouponEvent_20250128-Manufac...,LUID_00003H1O_BondCouponEvent_20250228,LUID_00003H1O_BondCouponEvent_20250228-Manufac...,LUID_00003H1O_BondCouponEvent_20250328,LUID_00003H1O_BondCouponEvent_20250328-Manufac...,LUID_00003H1Q_FlexibleRepoCashFlowEvent_2025-0...,LUID_00003H1Q_FlexibleRepoCollateralEvent_2025...,LUID_00003H1Q_FlexibleRepoInterestPaymentEvent...,LUID_00003H1Q_MaturityEvent_20250331,LUID_00003H1O_BondCouponEvent_20250428,LUID_00003H1O_BondCouponEvent_20250528
generated_event.instrument_identifiers.Instrument/default/ClientInternal,Seller-TermFlexRepo,BondCollateral,Seller-TermFlexRepo,BondCollateral,Seller-TermFlexRepo,BondCollateral,Seller-TermFlexRepo,Seller-TermFlexRepo,Seller-TermFlexRepo,Seller-TermFlexRepo,Seller-TermFlexRepo,BondCollateral,BondCollateral


## Load manual closure events as corporate actions

`FlexibleRepoFullClosureEvent` and `FlexibleRepoPartialClosureEvent` events are not automatically emitted by LUSID and so must be loaded into a portfolio's corporate action source manually. See https://support.lusid.com/docs/handling-instrument-events-for-repurchase-agreements#appendix-triggering-a-partial-or-full-closure.

### Load `FlexibleRepoFullClosureEvent`

To demonstrate, uncomment this and create the `OpenRepo` instruments and portfolios, above.

In [47]:
# def full_closure_open_repo(luid, date):
    
#     ca_request = lm.UpsertInstrumentEventRequest(
#         # A corporate action event must have a unique identifier within its corporate action source
#         instrument_event_id = f"{luid}-{date}",
#         instrument_identifiers = {f"Instrument/default/LusidInstrumentId": luid },
#         description = f"{luid}-{date}",
#         instrument_event = lm.FlexibleRepoFullClosureEvent(
#             instrument_event_type = "FlexibleRepoFullClosureEvent",
#             entitlement_date = to_date(date),
#             settlement_date = to_date(date)
#         )
#     )
    
#     # Upsert corporate actions to LUSID as instrument events
#     try:
#         ca_response = corporate_action_sources_api.upsert_instrument_events(
#             scope = module_scope,
#             code = module_code,
#             upsert_instrument_event_request = [ca_request]
#         )
#         print(ca_response.failed) if ca_response.failed else print("Success")
#     except lu.ApiException as e:
#         print(e)

In [48]:
# full_closure_open_repo(luid_dict["Buyer-OpenFlexRepo"], "2025-01-18")

In [49]:
# for port in ports:
#     print(f"\n{port}")
#     get_portfolio_holdings(port, "2025-01-17 00:00:00") # Day before
#     get_portfolio_holdings(port, "2025-01-18 00:00:00") # Full closure date
#     get_output_transactions(port, "2025-01-01", "2025-07-30")
#     query_instr_events(port)